In [1]:
import kagglehub

In [2]:
from kagglehub import KaggleDatasetAdapter

In [3]:
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.


In [4]:
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fashion-product-images-small


Load and setup the model

In [5]:
import tensorflow
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import GlobalMaxPooling2D
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

In [6]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
model = ResNet50(weights='imagenet',include_top=False, input_shape=(224,224,3))
model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [8]:
model = tensorflow.keras.Sequential([
    model,
    GlobalMaxPooling2D()
])

In [9]:
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 2048)           │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

None


Extract Image Features

In [10]:
import os
filenames = []
images_path = "/kaggle/input/fashion-product-images-small/images"
for files in os.listdir(images_path):
  filenames.append(os.path.join(images_path,files))
print(len(filenames))
print(filenames[:10])

44441
['/kaggle/input/fashion-product-images-small/images/31973.jpg', '/kaggle/input/fashion-product-images-small/images/30778.jpg', '/kaggle/input/fashion-product-images-small/images/19812.jpg', '/kaggle/input/fashion-product-images-small/images/22735.jpg', '/kaggle/input/fashion-product-images-small/images/38246.jpg', '/kaggle/input/fashion-product-images-small/images/16916.jpg', '/kaggle/input/fashion-product-images-small/images/52876.jpg', '/kaggle/input/fashion-product-images-small/images/39500.jpg', '/kaggle/input/fashion-product-images-small/images/44758.jpg', '/kaggle/input/fashion-product-images-small/images/59454.jpg']


In [ ]:
img = image.load_img(path,target_size=(244,244))
img_array = image.img_to_array(img)
print("img_array shape", img_array.shape)

expanded_img_array = np.expand_dims(img_array, axis = 0)
print("Expanded dim: ", expanded_img_array.shape)

In [12]:
import os
import tensorflow as tf
import numpy as np

# 1. Collect file paths
images_path ="/kaggle/input/fashion-product-images-small/images"
filenames = [os.path.join(images_path, f) for f in os.listdir(images_path)]

# 2. Optimized image loader function
def load_and_preprocess(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    # Replace with your backbone's specific preprocess_input if needed:
    # from tensorflow.keras.applications.resnet50 import preprocess_input
    img = preprocess_input(img)
    return img

# 3. Build parallelized tf.data pipeline
BATCH_SIZE = 64  # Adjust to 64 if you encounter GPU Out-Of-Memory

dataset = tf.data.Dataset.from_tensor_slices(filenames)
dataset = (dataset
           .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(buffer_size=tf.data.AUTOTUNE))

# 4. Extract all features in batches on GPU
raw_features = model.predict(dataset, verbose=1)

# 5. Vectorized L2 normalization (entire matrix at once)
norms = np.linalg.norm(raw_features, axis=1, keepdims=True)
feature_list = raw_features / np.maximum(norms, 1e-12)

695/695 ━━━━━━━━━━━━━━━━━━━━ 251s 346ms/step


In [13]:
import numpy as np
import pickle

# 1. Save embeddings as a compressed binary NumPy file
np.save('embeddings.npy', feature_list)
# Or compressed (saves disk space):
# np.savez_compressed('embeddings.npz', embeddings=feature_list)

# 2. Save filenames list using pickle
with open('filenames.pkl', 'wb') as f:
    pickle.dump(filenames, f)

In [ ]:
from google.colab import files

files.download('embeddings.npy')

In [ ]:
# Load embeddings
feature_list = np.load('embeddings.npy')

# Load filenames
with open('filenames.pkl', 'rb') as f:
    filenames = pickle.load(f)